# Ensemble Methods - Overview

## First Part - Necessary Code

In [13]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from auxiliary_functions import chart

In [14]:
# Import company colors
with open('plot_colors.json', 'r') as pc:
    color_dict = json.load(pc)

c_light, c_dark, c_blue, theme_flowchart = color_dict['color_light'], color_dict['color_dark'], color_dict['color_blue'], color_dict['theme_flowchart']

# Define color map for plotting
#color_map = ListedColormap([c_blue, c_light, c_dark])

In [15]:
# Generate Data for plots
np.random.seed(42)
X = np.random.rand(100, 1) - 0.5
y = 3*X[:, 0]**2 + 0.05 * np.random.randn(100)

In [16]:
# Train model with 3 estimators and learning rate 1
gbrt_3 = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0, random_state=42)
gbrt_3.fit(X, y)

# Train model with 200 estimators and learning rate 0.1
gbrt_200 = GradientBoostingRegressor(max_depth=2, n_estimators=200, learning_rate=0.1, random_state=42)
gbrt_200.fit(X, y)


def plot_ensemble(X, y, models, figsize=(15,5)):
    
    axis_limit = [-.5, 0.5, -0.1, 0.8]
    
    x1 = np.linspace(axis_limit[0], axis_limit[1], 500)
    
    y_pred_3 = models[0].predict(x1.reshape(-1, 1))
    y_pred_200 = models[1].predict(x1.reshape(-1, 1))
    
    fig, ax = plt.subplots(1, 2, figsize=figsize, sharey=True)
    sns.set_context("talk", font_scale=0.8)
    
    ax[0].scatter(X, y, c=c_dark, s=20)
    ax[0].plot(x1, y_pred_3, c=c_light, linewidth=2.5, label="Ensemble predictions")
    ax[0].set(xlabel='X', ylabel='y', title='learning_rate=1.0, n_estimators=3')
    ax[0].axis(axis_limit)
    
    ax[1].scatter(X, y, c=c_dark,  s=20)
    ax[1].plot(x1, y_pred_200, c=c_light, linewidth=2.5, label="Ensemble predictions")
    ax[1].set(xlabel='X', title='learning_rate=0.1, n_estimators=200')
    ax[0].axis(axis_limit)
    
    ax[0].legend()
    ax[1].legend()

In [17]:
# Train test split
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=49)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120, random_state=42)
gbrt.fit(X_train, y_train)

errors = [mean_squared_error(y_val, y_pred) for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators, random_state=42)
gbrt_best.fit(X_train, y_train)

min_error = np.min(errors)


def plot_early_stopping(X, y, model, figsize=(15,5)):
    
    axis_limit = [-.5, 0.5, -0.1, 0.8]
    
    x1 = np.linspace(axis_limit[0], axis_limit[1], 500)
    
    y_pred_best = model.predict(x1.reshape(-1, 1))
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    sns.set_context("talk", font_scale=0.8)
    
    ax[0].plot(errors, c=c_blue, marker='o', markersize=5)
    ax[0].plot([bst_n_estimators, bst_n_estimators], [0, min_error], "k--")
    ax[0].plot([0, 120], [min_error, min_error], "k--")
    ax[0].text(gbrt_best.n_estimators_, min_error*1.2, "Minimum", ha="center")
    ax[0].set(xlabel='Number of trees', ylabel='Error', title='Validation Error')
    ax[0].axis([0, 120, 0, 0.01])
    
    ax[1].scatter(X, y, c=c_dark,  s=20)
    ax[1].plot(x1, y_pred_best, c=c_light, linewidth=2.5)
    ax[1].set(xlabel='X', ylabel="y", title=f"Best model ({gbrt_best.n_estimators_} trees)")
    ax[1].axis(axis_limit)

## Second Part - Lecture

<div class="slide-title">

# Ensemble Methods - Overview
    
</div>    

Notes:  
We will cover ensemble methods in a more general way than when we spoke about each algorithm separately.  
More like an overview than going into detail.  

## Agenda

1. Recap CARTs
2. How and why does Bagging work
3. Different kinds of Ensemble Methods  
    - Bagging (Random Forests, Extra Trees)  
    - Boosting  
    - Stacking  
4. Calibration
5. Interpretability


Notes:  
"recap" in case random forests have been shown the day before.

## 1. Recap CART - Classification and regression trees

<div class="slide-chapter-split">
    <div class="title-chapter">Recap CART - Classification and regression trees</div>
    <div class="image-container">
        <img src="../images/ensemble_methods/img_p2_1.png">
    </div>
</div>

Notes: CART is the algorithm used for the decision tree algorithm.

### CART (Classification and Regression Tree)

<div class="group">
    <div class="text_70">

**General:**

* Finding the optimal split of the data is NP-complete.
* CART is a greedy procedure.
* The split function chooses **best feature and best value to split on**.

**Cost functions:**

* Regression $\rightarrow J(b)=\frac{1}{n}\sum_{i=1}^{n}{(y_{i}-\hat{y_{i}})^2}$ (MSE)
* Classification $\rightarrow I_{G}(p)= 1-\sum_{i=1}^{J}\,p_{i}^{2}$ (Gini impurity)
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/decision_tree/img_p13_1.png">
    </div>
</div> 

<div class="alert alert-block alert-info">
<b>Note:</b> 
NP-complete (nondeterministic polynomial-time complete) defines a problem which is easy to understand, easy to verify but difficult to solve e.g. a jigsaw puzzle.
</div>

Notes:  
J(b) is cost function for regression with b parameters
I_G(p) is the gini index which depends on the probability p by which you get a certain class for a given purity.  
"NP complete" is fixed term in computational complexity (like in this order P: polynomial time; NP: nondeterministic polynomial time; NP-complete: nondeterministic polynomial time complete and NP-hard). They order how easy to solve and how easy to verify a problem/solution is.

The complexity is oftern pharsed in big-O-notation, e.g. O(n) or O(n²), O(2^k) ... (required time is proportional to n or n², e.g.) 
Here it is O(n * log(n))

### CART 

<div class="group">
    <div class="text_70">

**Regularisation:**

* pruning to avoid overfitting
* Grow a full tree and then prune e.g.
    - `max_leaves`
    - `max_depth`
    - `min_sample_size`
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/decision_tree/img_p13_1.png">
    </div>
</div> 


### CART - pros and cons

<br>

<table class="confusion_matrix" border="1" width="1800">
  <thead>
    <tr>
      <th><b>&nbsp;&nbsp;PROS&nbsp;&nbsp;</b></th>
      <th><b>&nbsp;&nbsp;CONS&nbsp;&nbsp;</b></th>
    </tr>
  </thead>
</table>

### CART - pros and cons

<br>

<table class="confusion_matrix" border="1" width="1800">
  <thead>
    <tr>
      <th><b>PROS</b></th>
      <th><b>CONS</b></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td class="smaller-font-size">white box model - easy to interpret</td>
      <td class="smaller-font-size">not very accurate ...</td>
    </tr>
    <tr>
      <td class="smaller-font-size">can handle mixed data, discrete and continuous</td>
      <td class="smaller-font-size">greedy nature of constructing</td>
    </tr>
    <tr>
      <td class="smaller-font-size">insensitive to transformations of data... split points based on ranking  </td>
      <td class="smaller-font-size">trees are unstable, small changes to the input can lead to large changes in tree structure</td>
    </tr>
    <tr>
      <td class="smaller-font-size">relative robust to outliers</td>
      <td class="smaller-font-size">tend to overfit (aka high variance estimators)</td>
    </tr>
  </tbody>
</table>

### Trees being unstable

**Q:** Should you trust and follow a single CART?

<div class="group">
    <div class="text_70">       
        
* Changes in your trees can lead to different structure and maybe different decisions
* As in investment, it is not advisable to lay all eggs in one basket
    </div>
    
    <div class="images_30">   
        <img src="../images/ensemble_methods/img_p7_1.png">
    </div>
</div>

### 2. Trees being greedy and overfitting

<div class="group">
    <div class="text_70">

How to find out what to order at the restaurant?
* **Try everything!** <br>... and then order what you liked best.
* **Ask your friends to try something**, <br>find out which they preferred more and try those maybe?

**Q:** Which do you find more feasible?
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/eating_cloud.png">
    </div>
</div> 


<center>
    <img src="../images/ensemble_methods/img_p9_1.png" position="center" size="cover">
</center>

Notes: Who wants to be a millionaire allows contestants to get help from a friend and help from the audience. Over the run of the show, which help type do you think performed better? by how much? The majority of the audience picked the right answer 91 percent of the time, while individual friends only did so 65 percent of the time. This is an example of “crowd wisdom”, it’s an idea, summarized in the 2004 book by James Surowiecki by the same name, which states that the aggregate information in a group often leads to a better decision than any single member of the group. Image licence: unknown

## Wisdom of Crowds Theory

* Diversity of opinions
* Independence of opinions
* Decentralisation
* Aggregation

Notes: Let’s see what are the requirements or assumptions of the Crowd Wisdom theory:  
1. Each person has personal background / information.  
2. People draw their own, personal conclusion.  
3. This refers to the organization. People do not just follow the swarm (which they could even if independant).    
4. Some mechanism exists to turn private judgements into a collective decision.  
Do you think they hold for both WWtBaM and Wikipedia? <CLICK>

<div class="slide-chapter">

## 3. How can we produce different models with the same data?

</div>


<div class="slide-chapter">

## I. Bagging

</div>

### Bagging

<span class="color-brand" style="font-weight:bold;">B</span>ootsprap <span class="color-brand" style="font-weight:bold;">agg</span>regat<span class="color-brand" style="font-weight:bold;">ing</span>

$$ f(x)= \frac{1}{m}\sum f_{m}(x)$$  
        
Training $m$ **different trees** on **different subsets** of data chosen randomly with replacement.

Notes:  
mean (instead of mode) when we have a regression problem.

### Bootstrapping



<div class="group">
    <div class="text_70">

**Intuition:** Do something "impossible" by creating more samples than theorectically available by...



...*sampling with replacement* 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/bootstrapping.png">
    </div>
</div> 




Notes:  
1. In Random Forest we select an instance and put it back (replacement) before we take another instance.  
2. We also select certain features which we also can use another time (replacement).  
3. We do this as often as we want to created different trees (and can get any number of trees).  

Bootstrapping is a statistical procedure that resamples a single dataset to create many simulated samples. In our context the dataset is split into a number of subsets similar to how bootstrapping is traditionally performed. Sampling with replacement ensures each bootstrap is independent from its peers, as it does not depend on previous chosen samples when sampling. Then, m models are fitted using the above m bootstrap samples and combined by averaging the output (for regression) or voting (for classification). https://statisticsbyjim.com/hypothesis-testing/bootstrapping/  
The larger the dataset the less likely that the same instance will be drawn again.

## Majority Voting - how it works

For **each observation** you have **different predictions** from **different models**

You need one prediction... so you aggregate

Aggregating the different models can be done in many ways e.g. averages, mode, median




### Majority Voting - how it works

For each observation you have different predictions. How do you aggregate this to get one value?

### Hard Voting


<div class="group">
    <div class="text_70">
takes the majority  
class 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/hard_voting.png">
    </div>
</div> 




### Soft Voting


<div class="group">
    <div class="text_70">
average of summed probability 
vectors  
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/hard_voting.png">
    </div>
</div> 




<div class="alert alert-block alert-info">
<b>Note:</b> 
    
Soft voting tends to outperform hard voting.

</div>

### Majority Voting - why it works

Consider random variables $X_i$ which are independent identically distributed (i.i.d):

- Their variance is $\text{Var} (X_{i})=\sigma^{2}$

- If $\bar{X}$ is the result of an ensemble method, we get this variance:   

- $\text{Var} (\bar{X})=\text{Var}\left(\frac{1}{n}\sum_{i=1}^{n}X_i\right) = \frac{1}{n^2} \cdot \sum_{i=1}^{n} \text{Var}\left(X_i\right) = \frac{\sigma^2}{n}$ (central limit theorem) which is $n$ times smaller.



- If the $X_i$ are note completely independent, but correlated by $\rho$, we get:

 $$ \text{Var}(\bar{X})=\rho\sigma^{2}+(1-\rho)\frac{\sigma^{2}}{n}$$

Note:  
Central limit theorem: When you take sufficiently large random samples from any population (regardless of its distribution), the distribution of the sample means will approximate a normal distribution with $sigma_{mean}^2$ = $sigma_{population}^2$ / n  (n is #obs of samples, not number of samples)  
We use x instead of y here just because it's general math and not restricted to ML.  

To get a small variance of the ensemble, we want  
1. small $\rho$ (only the right term remains...)  
2. large $n$ (...and becomes small.)  
3. learners which are at least slighty better than random (not recognizable from formula but see next slide)  
 (When they are too strong (low variance), they usually go along with a high bias which is not solved by EM.)  
 error = bias² + variance

First, recall that for any random variables $X_i$: $\text{Var}(\bar{X}) = \text{Var}(\frac{1}{n}\sum_{i=1}^n X_i) = \frac{1}{n^2}\text{Var}(\sum_{i=1}^n X_i)$  


Expand the variance of the sum: $\text{Var}(\sum_{i=1}^n X_i) = \sum_{i=1}^n \text{Var}(X_i) + \sum_{i\neq j} \text{Cov}(X_i, X_j)$  


Since all $X_i$ have the same variance $\sigma^2$, and there are $n$ terms: $\sum_{i=1}^n \text{Var}(X_i) = n\sigma^2$  


For the covariance terms, recall that $\text{Cov}(X_i, X_j) = \rho\sigma^2$ for $i \neq j$. There are $n(n-1)$ such terms: $\sum_{i\neq j} \text{Cov}(X_i, X_j) = n(n-1)\rho\sigma^2$  


Substituting back into the equation from step 2: $\text{Var}(\sum_{i=1}^n X_i) = n\sigma^2 + n(n-1)\rho\sigma^2$
Now, substituting this into the equation from step 1: $\text{Var}(\bar{X}) = \frac{1}{n^2}[n\sigma^2 + n(n-1)\rho\sigma^2]$  


Simplifying: $\text{Var}(\bar{X}) = \frac{\sigma^2}{n} + \frac{n-1}{n}\rho\sigma^2$  


Rearranging: $\text{Var}(\bar{X}) = \rho\sigma^2 + \frac{\sigma^2}{n} - \frac{\rho\sigma^2}{n}$  


Factoring out $\sigma^2$: $\text{Var}(\bar{X}) = \rho\sigma^2 + (1-\rho)\frac{\sigma^2}{n}$  

### Majority Voting - why it works  

$$Error = \underbrace{Bias^2}_{underfitting} + \underbrace{Variance}_{overfitting} + \underbrace{Noise}_{unpredictable}$$

Notes:  
Students might remember this Error Decomposition from Bias-Variance Tradeoff lesson.  
While the Bias is small for decision trees, the variance gets small by using many of them (RF). Unfortunately bootstrapping can lead to correlations between the trees.

### Majority Voting - why it works



<div class="group">
    <div class="text_70">

**Intuition:** at observation level not all classifiers will be wrong in the same way at the same time

**Prerequisites** for this to work:
* models need to be better than random, i.e. **weak learners**
* you want the models to be different and that get translated in statistics to non-correlated or **sufficiently independent** or .. diverse
* you need a **sufficient amount** of weak learners 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/multible_learners.png">
    </div>
</div> 


<div class="alert alert-block alert-info">
<b>Note:</b> 
    
A weak learner model has an average true positive rate slightly better than 50% and an average false positive rate slightly less than 50%.

</div>

Notes:  
This statement refers to weak classification learners.  
In context of regression weak learners: They should at least be slightly better than simple average or simple linear regression.  

### Majority Voting - why it works


<div class="group">
    <div class="text_70">

**What does it mean that the models are independent / different?**

If two weak learners were correlated, an observation is likely to result in the same prediction.
    </div>
    <br>
    <div class="images_30">
        <img src="../images/ensemble_methods/weak_learners.png">
        **<span class="color-brand">A weak learner is at least 51% right.</span>**
    </div>
</div> 

<div class="alert alert-block alert-info">
<b>Note:</b> 
    
This can be proven by math. Feel free :D to try.. or <br>
run 1000 coin toss experiments and aggregate.

</div>

### Example of weak learners

<div class="group">
    <div class="text">

$$m=1000, \ p=0.51$$
        
$$p(x<k) = cdf(k=500) = 27.4\%$$

        
Chances to be wrong with m = 1000 weak learners.
    </div>
    <div class="images">
        Cumulative distribution function (cdf)
        <img src="../images/ensemble_methods/img_p21_1.png" width=400>
    </div>
</div> 

Notes:  
unfair coin (51% heads). In this cdf we see how probable (y) it is to get x or less times Heads. (e.g. 510 is 50%) If learners would just guess (50/50), we would expect a probability of 50% for 500 Heads. But if the (1000) learners are slightly better, (51/49), the probability of 500 or less would just be 27.4% or:  The ensemble accuracy would already be 72,6%.

### Example of weak learners

<div class="group">
    <div class="text">

$$m=1000, \ p=0.55$$

$$p(x<k) = cdf(k=500) = 0.08\% $$
        
Chances to be wrong with m = 1000 weak learners.
    </div>
    <div class="images">
        Cumulative distribution function (cdf)
        <img src="../images/ensemble_methods/img_p22_1.png" width=400>
    </div>
</div> 

Notes: The question “answered” here is: If we have an unfair coin, that is a coin that shows HEADS 51% (p=.51) and TAILS only 49% of the time. Experiment: Let's flip the coin 1000 times (n=1000) and count how often we got HEADS (K) We would expect for each run of the experiment to get 510 HEADS (=expected value =n*p ) and 490 TAILS (sometimes a little more, sometimes a little less) Now, let's conduct the experiment a million times and remember how often we got heads(K). If we make a histogram for K (~p(x=k)), we would expect to see a bell-curve. Most of the time we get 510, almost as often 509 or 511, almost as often 508 or 512… with decreasing occurrences left and right of the expected 510. If we take the histogram start from the left and successively add the values we encounter, we get to the cpf, i.e. the cumulative probability function which shows us the probability of getting “less than k” observations. (Shown on this slide) In our example, the chance of getting less than 1001 Heads is? p=1. We cannot get more than 1000 Heads in 1000 throws. the chance of getting less than 540 heads is (according to the plot) ~98% (-> or if you look at it from the other way: the chances of getting more than 540 heads in our 1000 throws is already only about 2%) the chance of getting less than 510 heads is (according to the plot) =50% (because this is exactly the expected value!) To bring it back to the learners: Each Weak learner is basically a coin flip with a slightly unfair coin: if each of the 1000 weak learners has an accuracy of 51% (and they are all completely independent from each other). Making a prediction now corresponds to the experiment from above: we “flip” each weak learner and the result. We would expect (given the accuracy) to get for each experiment run (about) 510 right and 490 wrong answers. But – it could also be 505 (right)vs 495(wrong)… or 496(right) vs 504(wrong). If we apply majority voting, the first two (510 / 505 right) would make us take the correct prediction, whereas the last one (496 right) would make us take the wrong prediction. The magic point here is: how often are less than 500 of our models correct (or in a formula p(x<k=500)). From the shown graph we can see that this is 27,4%. SO If we select the majority answer from 1000 weak learners with an accuracy of just 51% -> we get an ensemble accuracy of 72,6% (= 100-27.4)! If we do the same thing, but we make our learners slightly better, i.e. they have an accuracy of 55% percent vs 51%, we get the cdf from the next slide. Then the chance of less than 500 of our models being correct (or in a formula p(x<k=500) is down to 0,08% -> so we get an ensemble accuracy of 99,92%  
In short: If the 1000 learners are even better (55/45), we already get an accuracy of 99.2%, wow.

### Reducing variance by aggregation

<center>
    <img src="../images/ensemble_methods/img_p14_1.png" position="center">
</center>

Notes: I hope you’re starting to find the Wisdom of Crowds as interesting as I do. This is also <CLICK> the underlying theory of reducing the variance by aggregation, which you might also known as the philosophical idea behind a ML algorithm called Random Forest. In which we partition the data to small subsets, make an estimation based on each small subset, and then aggregate the result to get a better estimation.

### Random forest

Random forest is a technique that tries to decorrelate the predictors

* randomly chosen **subset of input variables (features)**
    * leading to different features used at each node
     
* randomly chosen **subset of data cases (observations)**
    * leading to out of bag instances
         
Random forest gain in performance but lose in interpretability.

Notes:  
Any kind of model can be used as weak learner, however DT are gladly used as they generally have low bias and high variance, while variance is reduced by ensembling. Furthermore DT are sensitive to train data which uncorrelates the weak learners' result.  

Terminology: 
- sampling both, features & observations is called "random patches".  
- sampling on features only is called "Subspaces".

Remark:  
- RF can output pseudo feature importance by counting how often a feature was used if it was part of majority.

### Random Forest


<div class="group">
    <div class="text_70">
using several 
decision trees
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/randomforest_concept.png">
    </div>
</div> 




<div class="alert alert-block alert-info">
<b>Note:</b> 
    
Running same learning algorithms on different subsets can result in still highly correlated predictors.

</div>

### Extra-Trees - Extremely Randomized Trees

Introduce even more randomness
* For each node, random split points are generated (rather than exhaustively testing all possibilities).  
 The splitting point with the best outcome is used for that node.  
* *All* the data in train is used to build each split (no bootstrapping).  
 (But like in random forest, a subset of random features is used.)  
* The trees are usually not pruned (similar to random forest).


Notes:  
(This is a frequent implementation. Others might exist as well.)  
1.) increases bias but computationally much cheaper AND reduces correlation  
2.) in turn reduces the bias  
3.) resulting in a low bias

Alltogether:  
Ensembles help to reduce the variance IN CASE the base learners are uncorrelated.  
Therefore 1 helps BUT increases bias. 2+3 in turn reduce the bias.

### Accuracy vs variance 

**Accuracy**
* Random Forest and Extra Trees outperform Decision Trees

**Variance**
* Decision trees → high
* Random Forest → medium
* Extra Trees → low

Notes:  
bagging helps to decrease the variance (large n with uncorrelated $sigma$) / reduce overfitting of the base models    
boosting helps to decrease Bias (see below) / by reducing the bias of the base models  
stacking improves total error (variance + bias, but stacking is not focused on improving weak learners)


### Out-of-Bag Evaluation

* For each model, there are instances that are not trained on (because of the replacement). They are called out-of-bag instances (oob).
* oobs are used for evaluation without the need for a separate validation set (out of bag error)

Notes:  
As every single weak learner does not use *all* observations --> no additional validation set needed for evaluation.  
However, extra trees use the entire dataset. Here we cannot evaluate out of bag.  

This approach does not replace train test split but is used for hyperparameter tuning.

## Ensemble Learning: Leveraging Multiple Perspectives

<div class="group">
    <div class="text_70">

A collection of models working together to produce a single, stronger prediction.

**Key Strategies:**
* **<span class="color-brand">Bagging</span>**: Trains independent models (usually the same algorithm) on random subsets of data to **reduce variance** (overfitting).
* **<span class="color-brand">Boosting</span>**: Trains models **sequentially**, where each model corrects the errors of the previous one to **reduce bias** (underfitting).
* **<span class="color-brand">Stacking</span>**: Trains diverse types of models in parallel and uses a **meta-learner** to intelligently combine their outputs for optimal predictive power.

**Core Advantages:**
* **Improved Robustness**: Aggregating models reduces the risk of relying on a single "unlucky" or biased learner.
* **Better Generalization**: By balancing the bias-variance tradeoff, ensembles typically outperform individual models on unseen data.
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/more_opinions.png">
    </div>
</div>



Notes:  
This is an overview slide again for letting the students keeping track after bagging is finished.

<div class="slide-chapter">

## II. Boosting
    
</div>

### Boosting

<div class="group">
    <div class="text">

A greedy algorithm for **fitting adaptive basis function**

$$ f(x)=b_{0}~\ + \sum b_{m}\phi_{m}(x)$$

Where:
* $\phi_{m}$ are the weak learners
* $b_0$ and $b_m$ are weights
* the weak learner is applied **sequentially** to **weighted versions** of the data
* more **weight** is given to examples **misclassified** by earlier rounds
* the weak learner can be **any regressor or classifier** but it is common to use CART → decision/regression tree
    </div>
    <div class="images">
        <img src="../images/ensemble_methods/boosting_step.png" width="750">
    </div>
</div> 

### Boosting algorithm


INIT
* Every instance has the same weight

LOOP 
* (Weights of instances incorrectly predicted are **increased**.) 
* Model is **trained** 
* All models **predict** 
* The predictions are **weighted by their accuracy and added**

### Boosting

<div class="group">
    <div class="text">

We do not go back and adjust earlier parameters → method is called **forward stagewise additive modeling**
    </div>
    <div class="images">
        <img src="../images/ensemble_methods/boosting.png">
    </div>
</div>

Notes:  
instead of using several low biased trees as base learners in bagging,  
boosting uses biased stumps only (which have a high bias).  
Weighing of the incorrectly predicted instances and weighing of the learners are two different kind of shoes.  

### Well-known Boosting Algorithms

| Algorithm | Key Characteristics | Pros | Cons |
|-----------|---------------------|------|------|
| AdaBoost | - Focuses on misclassified samples<br>- Assigns weights to weak learners | - Simple and intuitive<br>- Effective for binary classification<br>- Less prone to overfitting | - Sensitive to noisy data and outliers<br>- Can be computationally expensive<br> |
| Gradient Boosting | - Instead of weighing difficult datapoint, GB cares for the *residuals*<br>- Uses gradient descent for minimization<br>- Builds trees sequentially | - Flexible and versatile<br>- Handles different loss functions<br>- Good performance on various tasks | - Can be slow to train<br>- Prone to overfitting if not tuned properly<br>- Requires careful parameter tuning |
| XGBoost | - Regularized form of Gradient Boosting<br>- Uses second-order gradients<br>- most widely used | - Fast and efficient (parallelization)<br>- Handles missing values well<br>- Built-in regularization | - Can be memory-intensive<br>- May overfit on small datasets<br>- Complex to tune |
| LightGBM | - Uses histogram-based algorithms<br>- Leaf-wise tree growth | - Fast training and prediction<br>- Low memory usage<br>- Handles large datasets efficiently | - May overfit on small datasets<br>- Sensitive to overfitting if not configured properly<br>- Less interpretable than simpler models |
| CatBoost | - Handles categorical features automatically<br>- Uses ordered boosting | - Excellent performance on categorical data<br>- Good default parameters<br>- Less prone to overfitting | - Can be slower than other algorithms<br>- May underperform on purely numerical data<br>- Limited hyperparameter tuning options |


Notes:  
While the difficult *instances* get higher weights for the next iteration, the weights of the *learners* depend on the algorithm: e.g. AdaBoost weights the learners according to its individual accuracy while Gradient Boosting weighs them equally with a very small weight to avoid overfitting.  
select according to categorical/numerical problem; data/RAM ratio; simplicity for a baseline; amount of hyperparameters: https://towardsdatascience.com/catboost-vs-lightgbm-vs-xgboost-c80f40662924/

<div class="slide-chapter">

## III. Stacking
    
</div>


### Stacking


<div class="group">
    <div class="text_70">
combining different  
models 
    </div>
    <br><br><br>
    <div class="images_30">
        <img src="../images/ensemble_methods/stacking.png">
    </div>
</div> 




Notes:  
- Model 1, 2, 3 generate output from train data. shape(X_train) = n observations, #features  
- This output has n oberservations and #models columns/features. This is intput into a "final estimator" (here: blending)
- However, training of the "final estimator" must not be done with the output of the same training data (overfitting)  
- Therefore take 
   - the output of a hold out dataset (= blending, a certain kind of stacking) not having been used for the base model training or
   - the out-of-fold data from cross validation     
 to train the "final estimator"


<div class="slide-chapter">

## 4. Classifier calibration

</div>

<div class="alert alert-block alert-info">
<b>Note:</b> 
    
Methods based on the values of the probabilities need to return calibrated probabilities.

</div>

Notes:  
Classifiers usually provide a probability for a class. But the probabilities are not absolutely correct and depend on the used model algorithm. To make them comparable, the predicted probabilities need to be harmonized / calibrated first.  
The calibration is based on the assumption that the average predicted probability in a bin matches the actual proportion of positive cases in that bin. (The dataset should be balanced within each bin.)

### Calibrated probabilities

<div class="group">
    <div class="text">
        
Fitting a regressor (called a calibrator) that maps the output of the classifier (decision_function or predict_proba) to a calibrated probability in [0, 1]

[sklearn - calibrated probabilities](https://scikit-learn.org/stable/modules/calibration.html)
    </div>
    <div class="images">
        <img src="../images/ensemble_methods/img_p56_3_1.png">
    </div>
</div>

Notes:  
Calibration procedure:  
1. consider the table: model predicted probability vs actual class  
2. sort it by predicted probability and create probability bins
3. add a column TP/(TP+FN)  
4. the values in this column should be the same as the mean of the probability ranges of the corresponding bin. As this is usually not the case, create a regressor that maps the initial wrong probabilities to the corrected ones.  
(There is a link at the end for calibration.)  

### Frequency of predicted probabilities

<div class="group">
    <div class="text">
        
confidence and falsehood vs. shyness and correctness

*example*
    </div>
    <div class="images">
        <img src="../images/ensemble_methods/img_p56_3_2.png">
    </div>
</div>

Notes:  
Frequency of predicted probabilities for different classifiers. (Not related to calibration)
Confident classifiers can be wrong and shy classifiers can be right. But it's a nice opportunity to ask the students if there is a relation to calibration.
  
U-shapes mean a confident classifier (although it can be wrong).  
Mountain shapes are a bit shy and predict a lot of 50-50 predictions.  

<div class="slide-chapter">

## 5. Interpretability
    
</div>

### Interpreting black box models (like ensemble models)
* Linear models are popular due to interpretability, but poor performance
* Conclusions on the importance of a variable should be based on models with good accuracy
* Some of the approaches for measuring the effect of variables:
    * partial dependence plot
    * permutation feature importance


Notes:  
Why do we speak about interpretability here? Because ensemble methods are black box models but we even though want to know the impact of features.  
The two most important will be shown here: PDP and PFI  

### Partial dependence plots - Pros

<div class="group">
    <div class="text">
        
* Intuitive: the partial dependence function at a particular feature value represents the average prediction if we force all data points to assume that value
* causal interpretation (for the model)

    </div>
    <div class="images">
        <img src="../images/ensemble_methods/img_p59_1.png">
    </div>
</div>

Notes:  
1. train your model  
2. exchange one feature with one certain value only, say "0", and do the predictions  
3. This results in a lot of (n) 'number of bicycle' predictions: take the average for the figure on the left    
4. do this for all desired temperatures (not only for "0")  
 The given densities are based on the actual number of instances at that temperature (support) and provide a kind of reliability at a certain area.  
5. Do the same for the other features.  

Now you can see the impact of all features on the outcome.

### Partial dependence plots - Cons

<div class="group">
    <div class="text">

* maximum 2 features per plot
* assumption of independence between features
* Some PDPs omit showing feature value distribution, risking misinterpretation.

    </div>
    <div class="images">
        <img src="../images/ensemble_methods/img_p59_1.png">
    </div>
</div>

### Permutation feature importance - Pros

<div class="group">
    <div class="text">

* nice interpretation: the loss if the feature is removed
* it takes into account the interaction with other features
* does not require retraining the model

    </div>
    <div class="images">
        <img src="../images/ensemble_methods/img_p60_1.png">
    </div>
</div>

Notes:  
1. Create the model as usual and determine the model's performance (accurcy, MSE, ...)  
2. Shuffle the values of a certain column, do the prediction and calculate the increase (worsening).   
3. Do this several times to get a range of importances.  

The larger the increase, the larger the impact of this feature.  
Shuffling is like taking out the corresponding feature  



### Permutation feature importance - Cons

<div class="group">
    <div class="text">

* linked to the error of model
* doesn’t deal well with correlated features
* lacks directional insight

    </div>
    <div class="images">
        <img src="../images/ensemble_methods/img_p60_1.png">
    </div>
</div>

Notes:  
"linked to error": if the model under- or overfits, the results might be misleading  
"Lacks directional insight": in which direction will the output be going if a feature is modified  


## References

* Machine learning a probabilistic perspective - Kevin p. Murphy
* https://arxiv.org/pdf/1106.0257.pdf
* https://arxiv.org/pdf/1307.6522.pdf
* [Voting classifier blogpost](https://medium.com/@sanchitamangale12/voting-classifier-1be10db6d7a5)
* [Difference between extra trees and random forest](https://quantdare.com/what-is-the-difference-between-extra-trees-and-random-fo)
* [Explaining Adaboost, also including bagging and stacking](https://www.digitalocean.com/community/tutorials/adaboost-optimizer)
* [Videos about calibration](https://classifier-calibration.github.io/videos/)  
* [Calibration with sklearn](https://machinelearningmastery.com/calibrated-classification-model-in-scikit-learn/)  
* [Interpretable ML book](https://christophm.github.io/interpretable-ml-book)
* [Partial dependency plots](https://www.kaggle.com/dansbecker/partial-dependence-plots)
* &rarr; [SHapley Additive exPlanations](https://shap.readthedocs.io/en/latest/) &larr;